## Tools
-  Tools are runnable, these have `invoke` method.
- In LangChain, there are `built-in` and `custom` tools.

### Buil-in Tools
#### DuckDuckGo Search

In [7]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()
results = search_tool.invoke('ipl news')

print(results)

1 week ago - Stay updated with the latest IPL 2026 news, announcements, and match reports. Get all the information you need on IPLT20. The official IPL website with live scores, match updates, team rankings, all the latest news, and videos . Follow your favorite teams and players! 5 days ago - IPL 2026CSK in 2026: A season of stagnation amid flickers of hope 5 days ago - The IPL 2026 will start on March 28, with the tournament scheduled to run for more than two months. 1 hour ago - Stay updated with today's cricket news, live scores, and expert analysis. Covering the IPL 2026, Test matches, and domestic leagues worldwide. Get breaking stories, player interviews, and match reports from the world's leading cricket site.


In [8]:
results = search_tool.invoke('top news in India today')
print(results)

3 hours ago ... Latest News · Former AIADMK MLAs facing disqualification proceedings meet T.N. · Supreme Court mulls larger Bench reference in Meghalaya honeymoon murder case. 9 hours ago ... India Today LIVE TV: Trump News | US- Iran War | PM Modi in Australia | Hormuz | Ram Mandir Watch India Today LIVE TV for uninterrupted coverage of the ... 15 Jun 2026 ... India Today was live. Jun 15, 2026· . . #LIVE ... 7 hours ago ... Top stories · West Asia war LIVE: Iran Guards say hit U.S. bases in Kuwait, Bahrain, threaten to expand strikes · Modi calls for 'historic' Australia-India ... 2 days ago ... India's most trusted English news app. Rated #1 for trust among Indian news brands, Reuters Institute Digital News Report 2025. Get breaking news alerts, ...


In [9]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


#### Shell Tool
- Library `langchain-experimental` is needed for shell tool below.

In [10]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()
results = shell_tool.invoke('whoami') 
results

Executing command:
 whoami


c:\Users\koyel\Downloads\python_practice\ai_ml_learning_journey\nitish_singh\genai_notebooks\learning_env\Lib\site-packages\langchain_community\tools\shell\tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


'koyelpramanick\\koyel\r\n'

- Command `whoami` shows my name on current machine.

In [11]:
results = shell_tool.invoke('ls')
results

Executing command:
 ls


"'ls' is not recognized as an internal or external command,\r\noperable program or batch file.\r\n"

### Custom Tools
- LangChain provides option to write custom tools as per requirement.
- While calling tools, we send tool schema to llm (already present in built-in tools). We can create schema in our custom tools.
- There are multiple ways of writing custom tools.
#### Method - 1
- Docstring is highly recommended to add in tools bcz in future LLM will refer this docstring to understand working of the tool
- We'll see step by step how to write tools.
- First, we will just write a function, putting docstring.
- Then we will add type hint, tis will not enforce datatype of inputs, but developper will get type hint while code developping. It will help LLM to understand what type of data it will get in return from this tool.
- Docstring and type hinting are also recommended in case we are using AI (GitHUb copilot or other AI tools) to develop some usecase based on that codebase, in that case also this type hint and docstring will be useful.
- In third step we'll add tool decorator.

In [12]:
from langchain_core.tools import tool

# step 1 - create a function
def multiply(a, b):
    """Multiply two numbers"""
    return a*b

# step 2 - add type hints
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a*b

# step 3 - add tool decorator
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a*b

In [13]:
result = multiply.invoke({'a': 3, 'b':5})
print(result)

15


In [14]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


#### Method - 2
- Here to use `StructuredTool`.
- Here we strictly enforce argument type to tool, which was not possible in earlier method.

In [16]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

def multiply_func(a: int, b: int) -> int:
    return a*b

multiply_tool = StructuredTool.from_function(
    func = multiply_func,
    name = "multiply",
    description = "Multiply two numbers",
    args_schema = MultiplyInput
)

result = multiply.invoke({'a': 3, 'b':5})
print(result)

print(multiply.name)
print(multiply.description)
print(multiply.args)


15
multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


C:\Users\koyel\AppData\Local\Temp\ipykernel_14216\3515107248.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a: int = Field(required=True, description="The first number to add")
C:\Users\koyel\AppData\Local\Temp\ipykernel_14216\3515107248.py:6: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b: int = Field(required=True, description="The second number to add")


#### Method - 3: 
- Here to use `BaseTool` class.
- Name of run method should be exactly `_run`.
- Can create `async version` of tool in this method, which was not possible in previous methods.

In [17]:
from langchain.tools import BaseTool
from typing import Type

# arg schema using pydantic
class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"
    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a*b

multiply_tool = MultiplyTool()

result = multiply_tool.invoke({'a': 3, 'b': 5})

C:\Users\koyel\AppData\Local\Temp\ipykernel_14216\3854103587.py:6: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a: int = Field(required=True, description="The first number to add")
C:\Users\koyel\AppData\Local\Temp\ipykernel_14216\3854103587.py:7: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b: int = Field(required=True, description="The second number to add")


- There are lots of built-in tools in LangChin. All are not required ti learn at one go. Tools can be visited as per requirement in project.
- Read LangChain documentation.

- TODO:
    - Paste documentation link.

## ToolKit
- When we have different tools related to same software or any other same group user e.g. suppose we have tools to read files from Google drive and to upload files to Google drive, then as these two tools are related to Google drive only, we can keep them in ToolKit.
- Let's see an example to make a ToolKit of custom tools.

In [18]:
from langchain_core.tools import tool

@tool
def add(a: int, b:int) -> int:
    """Add two numbers"""
    return a+b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a*b

class MathToolkit:
    def get_tools(self):
        return [add, multiply]

toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)


add => Add two numbers
multiply => Multiply two numbers


## Tool Calling

In [19]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [20]:
# tool create
@tool
def multiply(a: int, b: int) -> int:
    """Given 2 numbers a and b this tool returns their product"""
    return a*b

In [21]:

print(multiply.invoke({'a': 3, 'b': 5}))

15


In [22]:
multiply.name

'multiply'

In [23]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

In [24]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [25]:
# calling LLM
llm = ChatOpenAI()

- Now we ill do tool binding.
- Here is only one tool to bind. 
- Add ohther tools by putting `comma` separator if there are more than one tool.
- There are few LLMs which are able to do tool binding (i.e. able to work with toosls).

In [27]:
llm_with_tools = llm.bind_tools([multiply]) 

Now we will see tool calling where necessary. E.g. - in the below qestion `Hi how are you?` does not require any tool call. So here no tool calling will happen. Argument `tool_calls` is empty.

In [28]:
llm_with_tools.invoke('Hi how are you?') 

AIMessage(content="Hello! I'm here and ready to help you with anything you need. How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 59, 'total_tokens': 82, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DzgvN3uvKnAqjD7F1J08VzkTBB078', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f4691-aa07-7d10-8ead-c1d99c6b0125-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 59, 'output_tokens': 23, 'total_tokens': 82, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

But for question `Can you multiply 3 with 10?` tool call is needed. So here tool call is happening.

- Here LLM can only suggest tools required, it will not call tool. There can be confusion from the name tool call that LLM will call tools, but that's not correct.
- If LLM will execute tool calling that will be risky, here should be accountability, which LLM can't provide.
- So, actual calling of tools will be on developper only.

In [29]:
llm_with_tools.invoke('Can you multiply 3 with 10?')

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 63, 'total_tokens': 80, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DzgvVMbuKc9kMbzLNKqWxoLg4LqBw', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f4691-cf79-7882-b047-730401ceeded-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'call_7VaiBsU1QWr7O9QVZ99HTu0a', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 63, 'output_tokens': 17, 'total_tokens': 80, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [30]:
llm_with_tools.invoke('Can you multiply 3 with 10?').tool_calls

[{'name': 'multiply',
  'args': {'a': 3, 'b': 10},
  'id': 'call_Tu5gV1aCoaZCKitQKePXhYFm',
  'type': 'tool_call'}]

In [31]:
llm_with_tools.invoke('Can you multiply 3 with 10?').tool_calls[0]

{'name': 'multiply',
 'args': {'a': 3, 'b': 10},
 'id': 'call_WbzuJJnSLIIMONPCGi87hXCg',
 'type': 'tool_call'}

In [32]:
# tool execution
result = llm_with_tools.invoke('Can you multiply 3 with 10?')
result.tool_calls[0]['args']

{'a': 3, 'b': 10}

In [33]:
multiply.invoke(result.tool_calls[0]['args'])

30

- Here below we will get `ToolMessage` when we are sending  full tool_calls[0] instead of only arguments.
- This is another type of message along with `HumanMessage`, `AIMessage`, `SystemMessage`.

In [34]:
multiply.invoke(result.tool_calls[0])

ToolMessage(content='30', name='multiply', tool_call_id='call_k24DopG7WrJCRzGTP3ESbenN')

In [35]:
# query
query = 'Can you multiply 3 with 10?'

# maintain conversation history
messages = [HumanMessage(query)]
print(messages)

[HumanMessage(content='Can you multiply 3 with 10?', additional_kwargs={}, response_metadata={})]


In [36]:
result = llm_with_tools.invoke(messages)
print(result)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 63, 'total_tokens': 80, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DzgvwdrBRplMIs4N6dwV6hlSBfEiF', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019f4692-3b1b-7f91-b650-3df5250c87fb-0' tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'call_iSiWmiXjB7EpKTI3vWkJJpv5', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 63, 'output_tokens': 17, 'total_tokens': 80, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [37]:
# append result with previous message
messages.append(result)
print(messages)

[HumanMessage(content='Can you multiply 3 with 10?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 63, 'total_tokens': 80, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DzgvwdrBRplMIs4N6dwV6hlSBfEiF', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f4692-3b1b-7f91-b650-3df5250c87fb-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'call_iSiWmiXjB7EpKTI3vWkJJpv5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 63, 'output_tokens': 17, 'total_tokens': 80, 'input_token_details': {'audio': 0, 'cache_r

In [38]:
# call tool
tool_result = multiply.invoke(result.tool_calls[0])

In [39]:
# append result with previous message
messages.append(result)
print(messages)

[HumanMessage(content='Can you multiply 3 with 10?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 63, 'total_tokens': 80, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DzgvwdrBRplMIs4N6dwV6hlSBfEiF', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f4692-3b1b-7f91-b650-3df5250c87fb-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'call_iSiWmiXjB7EpKTI3vWkJJpv5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 63, 'output_tokens': 17, 'total_tokens': 80, 'input_token_details': {'audio': 0, 'cache_r

In [40]:

print(multiply.invoke(result.tool_calls[0]))

content='30' name='multiply' tool_call_id='call_iSiWmiXjB7EpKTI3vWkJJpv5'


In [41]:
print(multiply.invoke(result.tool_calls[0]).content)

30


- Try above with new query.

In [42]:
query = 'Can you multiply 3 with 1000?'
messages = [query]
print(messages)

['Can you multiply 3 with 1000?']


In [43]:
result = llm_with_tools.invoke(messages)
print(result)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 64, 'total_tokens': 82, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Dzh5kMsoStywBTHonBipJnnF3u2bg', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019f469b-8184-7d30-9018-8386f9da1234-0' tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_DIkn1j0aImbQ18aJozeAfh1J', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 64, 'output_tokens': 18, 'total_tokens': 82, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [44]:
messages.append(result)
print(messages)

['Can you multiply 3 with 1000?', AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 64, 'total_tokens': 82, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Dzh5kMsoStywBTHonBipJnnF3u2bg', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f469b-8184-7d30-9018-8386f9da1234-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_DIkn1j0aImbQ18aJozeAfh1J', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 64, 'output_tokens': 18, 'total_tokens': 82, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}

In [45]:
tool_result = multiply.invoke(result.tool_calls[0])
print(tool_result)

content='3000' name='multiply' tool_call_id='call_DIkn1j0aImbQ18aJozeAfh1J'


In [46]:
messages.append(result)
print(messages)

['Can you multiply 3 with 1000?', AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 64, 'total_tokens': 82, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Dzh5kMsoStywBTHonBipJnnF3u2bg', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f469b-8184-7d30-9018-8386f9da1234-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_DIkn1j0aImbQ18aJozeAfh1J', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 64, 'output_tokens': 18, 'total_tokens': 82, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}

In [47]:

print(multiply.invoke(result.tool_calls[0]))

content='3000' name='multiply' tool_call_id='call_DIkn1j0aImbQ18aJozeAfh1J'


In [48]:
print(multiply.invoke(result.tool_calls[0]).content)

3000


### Currency Conversion Tool

In [61]:
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    This function fetches the currency conversion factor between a given base currency and a target currency
    """
    url = f"https://v6.exchangerate-api.com/v6/3d7ecf69d1bc6db9a78939f5/pair/{base_currency}/{target_currency}" # write exchange rate url
    response = requests.get(url)
    return response.json()

In [62]:
get_conversion_factor.invoke({'base_currency': 'USD', 'target_currency': 'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1783555202,
 'time_last_update_utc': 'Thu, 09 Jul 2026 00:00:02 +0000',
 'time_next_update_unix': 1783641602,
 'time_next_update_utc': 'Fri, 10 Jul 2026 00:00:02 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.6063}

In [60]:
@tool
def convert(base_currency_value: int, conversion_rate: float) -> float:
    """
    Given a currency conversion rate this function calculates the target currency value from a given base currency value
    """
    return base_currency_value*conversion_rate

In [63]:
convert.invoke({'base_currency_value': 10, 'conversion_rate': 85.16})

851.5999999999999

In [65]:
# LLM calling
llm = ChatOpenAI()

# tool binding
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

messages = [HumanMessage('What is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR?')]
ai_message = llm_with_tools.invoke(messages)
ai_message

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 128, 'total_tokens': 187, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DzhNLM0COcu3FTWyA9mJDn0h8BWeJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f46ac-2584-71b3-bd64-493f320fd12a-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_eQlNfr4CKGzsziKYI0u6yN9L', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 10, 'conversion_rate': 74.5}, 'id': 'call_O8YTaXyKijE8zQdwd2jseYTh', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_token

In [66]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_eQlNfr4CKGzsziKYI0u6yN9L',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10, 'conversion_rate': 74.5},
  'id': 'call_O8YTaXyKijE8zQdwd2jseYTh',
  'type': 'tool_call'}]

- LLM tried to tackle both queries together, so it didn't use latest conversation rate. We are seeing conversion rate that LLM fetched from its parametric knowledge.
- To resolve this issue `injected tool argument` is used, where fetched value from earlier tools will be used, LLM does not use convertion rate from its parametric knowledge.

In [77]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    This function fetches the currency conversion factor between base currency and a target currency
    """
    url = f"https://v6.exchangerate-api.com/v6/3d7ecf69d1bc6db9a78939f5/pair/{base_currency}/{target_currency}" # write exchange rate url
    response = requests.get(url)
    return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """
    Given a currency conversion rate this function calculates the target currency value from a given base currency value
    """
    return base_currency_value*conversion_rate

# tool binding
llm = ChatOpenAI()
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

messages = [HumanMessage('What is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR?')]
ai_message = llm_with_tools.invoke(messages)

In [71]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_N6G2VXQ69C68aBnFEvkoQckM',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'call_iAfiL5XpytkeS9cBMmAThfLl',
  'type': 'tool_call'}]

In [72]:
messages.append(ai_message)

In [73]:
import json

for tool_call in ai_message.tool_calls:
    print(tool_call)

    # execute the 1st tool and get the value of conversion rate
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        print('tool_message1:', tool_message1)

        # fetch this conversion rate
        print("Content of tool_message1:")
        print(tool_message1.content)

        print('Fetched conversion_rate:')
        print(json.loads(tool_message1.content)['conversion_rate'])

        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        print("Fetched conversion rate:", conversion_rate)

        # append this tool message to messages list
        messages.append(tool_message1)


    # execute the 2nd tool using the conversion rate from tool 1
    if tool_call['name'] == 'convert':

        # fetch the current argument
        tool_call['args']['conversion_rate'] = conversion_rate

        tool_message2 = convert.invoke(tool_call)
        print('tool_message2:', tool_message2)
        
        messages.append(tool_message2)

{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_N6G2VXQ69C68aBnFEvkoQckM', 'type': 'tool_call'}
tool_message1: content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1783555202, "time_last_update_utc": "Thu, 09 Jul 2026 00:00:02 +0000", "time_next_update_unix": 1783641602, "time_next_update_utc": "Fri, 10 Jul 2026 00:00:02 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 95.6063}' name='get_conversion_factor' tool_call_id='call_N6G2VXQ69C68aBnFEvkoQckM'
Content of tool_message1:
{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1783555202, "time_last_update_utc": "Thu, 09 Jul 2026 00:00:02 +0000", "time_next_update_unix": 1783641602, "time_next_update_utc": "Fri, 10 Jul 2026 00:00:02

In [74]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 121, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DzhSDM0K26SEaowy8F34rdZWIimzO', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f46b0-bdaf-73f2-a4cb-99dfb8fb533e-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_N6G2VXQ69C68aBnFEvkoQckM', 'type': 'tool_call'}, {'name': 'convert', 'arg

- Three types of Messages will be here.

In [75]:
print(messages) 

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 121, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DzhSDM0K26SEaowy8F34rdZWIimzO', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f46b0-bdaf-73f2-a4cb-99dfb8fb533e-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_N6G2VXQ69C68aBnFEvkoQckM', 'type': 'tool_call'}, {'name': 'convert', 'args

In [76]:
llm_with_tools.invoke(messages).content

'The conversion factor between USD and INR is 95.6063. \n\nBased on this conversion factor, 10 USD is equivalent to 956.06 INR.'

- TODO:
    - Try with INR to USD conversion question.